# Data Exploration: MNIST Binary Classification

Explores the binary MNIST data used for the QNN experiments: loading,
samples, feature distributions, quantum encoding, and summary statistics.

In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


## 1. Load MNIST Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_mnist_binary

X_train, y_train, X_test, y_test = load_mnist_binary(
    digit1=3, digit2=6, train_size=1000, test_size=200, data_seed=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## 2. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for row, label in enumerate((0, 1)):
    xs = X_train[y_train == label][:10]
    for col, img in enumerate(xs):
        axes[row, col].imshow(img.reshape(28, 28), cmap="gray")
        axes[row, col].axis("off")
axes[0, 0].set_title("digit 3", loc="left")
axes[1, 0].set_title("digit 6", loc="left")
plt.tight_layout()
plt.show()


## 3. Feature Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(X_train.ravel(), bins=50)
axes[0].set_title("All features (raw)")
axes[1].hist(X_train[y_train == 0].ravel(), bins=50, alpha=0.6, label="digit 3")
axes[2].hist(X_train[y_train == 1].ravel(), bins=50, alpha=0.6, label="digit 6")
for ax in axes[1:]:
    ax.set_title("Per class")
plt.tight_layout()
plt.show()


## 4. Quantum Data Encoding

In [ ]:
from src.data import prepare_features, encode_data_for_qnn

N_QUBITS = 8
X_train_r, X_test_r, pca_info = prepare_features(
    X_train, X_test, n_components=N_QUBITS, image_size=(4, 4)
)
angles = encode_data_for_qnn(X_train_r)
print(f"Reduced train: {X_train_r.shape} -> encoded angles {angles.shape}")
print(f"Angle range: [{angles.min():.4f}, {angles.max():.4f}]")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(angles.ravel(), bins=30)
ax.set_xlabel("Rotation angle (rad)")
ax.set_ylabel("Count")
ax.set_title("Distribution of encoded angles")
plt.show()


## 5. Average Images per Class

In [ ]:
avg3 = X_train[y_train == 0].mean(axis=0).reshape(28, 28)
avg6 = X_train[y_train == 1].mean(axis=0).reshape(28, 28)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(avg3, cmap="gray")
axes[0].set_title("Average digit 3")
axes[1].imshow(avg6, cmap="gray")
axes[1].set_title("Average digit 6")
plt.tight_layout()
plt.show()


## 6. Data Statistics Summary

In [ ]:
stats = {
    "Dataset": ["Training", "Test"],
    "Samples": [len(X_train), len(X_test)],
    "Pixels/image": [X_train.shape[1]] * 2,
    "Class ratio (6/3)": [
        round(float(y_train.mean()), 3),
        round(float(y_test.mean()), 3),
    ],
}
for key, values in stats.items():
    print(f"{key:16s} " + "  ".join(f"{v}" for v in values))


## 7. Conclusion

- Data is loaded seed-decoupled and reduced via a fixed
  low-dimensional PCA pipeline.
- Features are scaled to rotation angles in `[0, pi]`.
- The reduced dimensionality equals the qubit count (`n_components = n_qubits`).